<a href="https://colab.research.google.com/github/Innocent250/esm/blob/main/ngs_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NGS Amplicon Analysis Pipeline for CRISPR Genome Editing Assessment

**Author:** Innocent Byiringiro | University of Maryland, College Park | [ibyiring@umd.edu](mailto:ibyiring@umd.edu)
<br>**Lab:** [Qi Lab - Plant Genome Engineering](https://www.yipingqi.com/)

---


### How to Run This Notebook

**Option A: Google Colab (Recommended -- no local setup needed)**
1. Click the **"Open in Colab"** badge above
2. In Colab: `Runtime` > `Change runtime type` > select **Python 3** (GPU not required)
3. Upload your paired-end `.fastq.gz` files (file browser on the left, or mount Google Drive)
4. Prepare a barcode CSV mapping sample names to barcode sequences (template: `barcodes/hi_tom_barcodes.csv`)
5. Run cells sequentially from top to bottom, updating file paths and parameters as indicated
6. Download the zipped CRISPResso2 output at the end

**Option B: Local Jupyter**
```bash
git clone https://github.com/Innocent250/ngs-analysis-pipeline.git
cd ngs-analysis-pipeline
pip install -r requirements.txt
jupyter notebook NGS_Analysis_Pipeline.ipynb
```

### Input Requirements

| Input | Format | Description |
|-------|--------|-------------|
| Paired-end reads | `.fastq` or `.fastq.gz` | Illumina MiSeq R1 and R2 files |
| Barcode table | `.csv` | Columns: `Sample`, `Barcode_L`, `Barcode_R` |
| Amplicon sequence | Text string | Full reference amplicon (including primer regions) |
| Guide RNA sequence | Text string | 20-nt guide sequence (without PAM) |

### Associated Publication

> Byiringiro, I.\*, Contiliani, D.F.\*, Davies, C., Creste, S., & Qi, Y. *Targeting A/T Rich PAM Sites for Advanced Genome Engineering in Plants by Expanded CRISPR-Combo Systems.* (In preparation)

---
## Step 1: Environment Setup

This cell installs all required dependencies:
> **Note:** CRISPResso2 (installed in Step 4) now uses **fastp** internally for read processing (replacing the deprecated FLASH dependency). No separate FLASH installation is needed.

In [ ]:
#@title Environment Setup
# ============================================================
# 1a. Set up working directory
#     All outputs will be saved under WORK_DIR.
#     On Colab this defaults to /content; change if running locally.
# ============================================================
import subprocess
import os
import sys
import gzip
import collections
from tqdm import tqdm
import pandas as pd

WORK_DIR = '/content'  #@param {type:"string"}
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
# ============================================================
# (Optional) Mount Google Drive if your data is stored there
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')

print('Environment setup complete.')

---
## Step 2: Read Merging

Paired-end Illumina reads (R1 and R2) overlap in the middle of the amplicon. This step merges them into a single, full-length sequence per read pair.

### How it works

The  function slides one read along the other to find the best-scoring overlap, then stitches the reads together.

**Key parameters you can adjust:**
| Parameter | Default | Description |
|-----------|---------|-------------|
| `overlap_len_require` | 10 bp | Minimum overlap length to accept a merge |
| `overlap_diff_percent_limit` | 60 | Minimum fraction of matching bases in the overlap |

> **Tip:** For Hi-TOM amplicons (typically 250-400 bp with 2x250 bp reads), the default parameters work well. If you see low merge rates, try decreasing `overlap_len_require` or `overlap_diff_percent_limit`.
>
> **Adjusting overlap range:** You can change `overlap_len_require`. For shorter amplicons with large overlaps, increase `overlap_len_require` to reduce false merges. For longer amplicons with small overlaps, decrease `overlap_len_require` to recover more reads.

- This step uses **fastp:**  [fastp](https://github.com/OpenGene/fastp) ([Chen et al., 2018](https://doi.org/10.1093/bioinformatics/bty560)), an ultrafast all-in-one FASTQ preprocessor that can merge reads, trim adapters, and generate QC reports in a single step. you can download the QC reports at this stage.

**Instructions:**
1. Upload your R1 and R2 FASTQ files to the Colab environment (into `WORK_DIR`, or mount Google Drive)
2. Update the file names below (paths are built automatically from `WORK_DIR`)

In [ ]:
#@title Merge with fastp

# # First, install fastp (only need to run once per session)

!wget -q http://opengene.org/fastp/fastp && chmod a+x ./fastp && mv fastp /usr/local/bin/

read_R1 = "os.path.join(WORK_DIR, 'read_R1')"  #@param {type:"string"}
read_R2 = "os.path.join(WORK_DIR, 'Read_R2')"  #@param {type:"string"}
merged_fastq = "os.path.join(WORK_DIR, 'merged_reads')" #@param {type:"string"}

!fastp \
    --in1 {read_R1} \
    --in2 {read_R2} \
    --merged_out {merged_fastq} \
    --out1 /dev/null1 --out2 /dev/null2 \
    --overlap_len_require 10 \
    --overlap_diff_percent_limit 60 \
    --merge \
    --html {os.path.join(WORK_DIR, 'fastp_report.html')} \
    --json {os.path.join(WORK_DIR, 'fastp_report.json')}

print(f'Merging complete. Output: {merged_fastq}')
print('QC report: fastp_report.html (download and open in browser)')

---
## Step 3: Demultiplex by Hi-TOM Barcodes

After merging, each read contains the full amplicon flanked by Hi-TOM barcodes. This step splits the merged reads into **individual sample FASTQ files** based on the barcode pairs assigned to each sample during PCR Round 2.

### How the Hi-TOM barcoding system works

In the Hi-TOM system ([Liu et al., 2019](https://doi.org/10.1007/s11427-019-9570-x)), each sample is assigned a unique combination of forward (F) and reverse (R) barcodes during the second round of PCR. With 12 forward barcodes (plate columns) and 8 reverse barcodes (plate rows), **96 unique sample barcodes** are possible per sequencing run.

```
Read structure after merging:
5'- [F barcode] - [bridging seq] - [TARGET AMPLICON] - [bridging seq] - [R barcode RC] -3'
```

The demultiplexer checks both orientations (forward-reverse and reverse-forward) to capture reads regardless of which strand was sequenced.

In [ ]:
#@title Demultiplexing function

import os

def reverse_complement(dna):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join([complement.get(base.upper(), 'N') for base in dna[::-1]])

def split_fastq(df, fastq, output_directory):
    """
    Demultiplex a merged FASTQ file by Hi-TOM barcode pairs.

    Expected forward structure:
        Barcode_L + insert + reverse_complement(Barcode_R)

    Expected reverse structure:
        Barcode_R + reverse_complement(insert) + reverse_complement(Barcode_L)

    Reverse-orientation reads are flipped back before writing,
    so CRISPResso receives reads in the forward amplicon orientation.
    """

    # Pre-compute barcode lookup
    barcodes = []
    for _, row in df.iterrows():
        sample = str(row['Sample'])
        left = row['Barcode_L'].upper()
        right_raw = row['Barcode_R'].upper()

        right_rc = reverse_complement(right_raw)
        left_rc = reverse_complement(left)

        barcodes.append({
            'sample': sample,
            'left': left,
            'right_rc': right_rc,
            'right_raw': right_raw,
            'left_rc': left_rc,
            'left_len': len(left),
            'right_len': len(right_raw),
        })

    # Create output directories and open file handles
    file_handles = {}
    for bc in barcodes:
        sample = bc['sample']
        out_dir = os.path.join(output_directory, sample)
        os.makedirs(out_dir, exist_ok=True)

        out_path = os.path.join(out_dir, f'{sample}.extendedFrags.fastq')
        file_handles[sample] = open(out_path, 'w')

    # Read FASTQ once and assign each read
    stats = {bc['sample']: 0 for bc in barcodes}
    stats_forward = {bc['sample']: 0 for bc in barcodes}
    stats_reverse = {bc['sample']: 0 for bc in barcodes}

    total_reads = 0
    unassigned = 0

    with open(fastq, 'r') as f:
        while True:
            header = f.readline().rstrip()
            if not header:
                break

            seq = f.readline().rstrip().upper()
            plus = f.readline().rstrip()
            qual = f.readline().rstrip()

            total_reads += 1
            assigned = False

            for bc in barcodes:
                ll = bc['left_len']
                rl = bc['right_len']

                # Forward orientation:
                # [Barcode_L]...[reverse_complement(Barcode_R)]
                if len(seq) >= ll + rl:
                    if seq[:ll] == bc['left'] and seq[-rl:] == bc['right_rc']:
                        trimmed_seq = seq[ll:-rl]
                        trimmed_qual = qual[ll:-rl]

                        file_handles[bc['sample']].write(
                            f"{header}\n{trimmed_seq}\n+\n{trimmed_qual}\n"
                        )

                        stats[bc['sample']] += 1
                        stats_forward[bc['sample']] += 1
                        assigned = True
                        break

                # Reverse orientation:
                # [Barcode_R]...[reverse_complement(Barcode_L)]
                if not assigned and len(seq) >= rl + ll:
                    if seq[:rl] == bc['right_raw'] and seq[-ll:] == bc['left_rc']:
                        trimmed_seq = reverse_complement(seq[rl:-ll])
                        trimmed_qual = qual[rl:-ll][::-1]

                        file_handles[bc['sample']].write(
                            f"{header}\n{trimmed_seq}\n+\n{trimmed_qual}\n"
                        )

                        stats[bc['sample']] += 1
                        stats_reverse[bc['sample']] += 1
                        assigned = True
                        break

            if not assigned:
                unassigned += 1

    # Close all file handles
    for fh in file_handles.values():
        fh.close()

    # Print demultiplexing summary
    print(f'\n{"="*50}')
    print('Demultiplexing Summary')
    print(f'{"="*50}')
    print(f'Total reads:      {total_reads:,}')
    print(f'Assigned:         {total_reads - unassigned:,} ({100*(total_reads-unassigned)/max(total_reads,1):.1f}%)')
    print(f'Unassigned:       {unassigned:,} ({100*unassigned/max(total_reads,1):.1f}%)')

    print('\nPer-sample counts:')
    for bc in barcodes:
        s = bc['sample']
        print(f'  {s:30s}  {stats[s]:>8,} reads')

    print('\nOrientation counts:')
    for bc in barcodes:
        s = bc['sample']
        print(
            f'  {s:30s}  '
            f'forward={stats_forward[s]:>8,}  '
            f'reverse={stats_reverse[s]:>8,}'
        )

    return stats

### Run Demultiplexing

**Prepare your barcode CSV file** with the following columns:

| Column | Description | Example |
|--------|-------------|---------|
| `Sample` | Your sample name | `Plant_1` |
| `Barcode_L` | Forward (left) barcode sequence | `gcttGCGTt` |
| `Barcode_R` | Reverse (right) barcode sequence | `ctgtGCGTt` |

> **Template:** A complete 96-well plate barcode CSV is available at `barcodes/hi_tom_barcodes.csv` in this repository. Copy it, replace the `Sample` column with your sample names, and remove unused wells.

**Update the file names below** (paths are built automatically from `WORK_DIR`):

In [ ]:
#@title Demultiplexing


# ============================================================
# UPDATE THESE for your experiment
# ============================================================
barcode_csv = "os.path.join(WORK_DIR, 'barcode.csv')"  #@param {type:"string"}
# merged_fastq was defined in Step 2 above

demux_output = "os.path.join(WORK_DIR, 'Split file')"  #@param {type:"string"}

reference_df = pd.read_csv(barcode_csv)

# Run demultiplexing (reads file once, exact barcode matching)
demux_stats = split_fastq(
    df=reference_df,
    fastq=merged_fastq,
    output_directory=demux_output
)

---
## Step 4: Install CRISPResso2

[CRISPResso2](https://github.com/pinellolab/CRISPResso2) ([Clement et al., 2019](https://doi.org/10.1038/s41587-019-0032-3)) quantifies genome editing outcomes from deep sequencing data:

- **Indel frequencies** (insertions and deletions around the cut site)
- **Allele frequency tables** (every unique sequence and its count)
- **Mutation profiles** (position-specific editing rates)
- **Publication-ready plots** (HTML and PNG)

> **Note:** CRISPResso2 v2.3+ uses **fastp** internally for read processing (replacing the previously required FLASH). No separate FLASH installation is needed.
>
> Installation takes 2-3 minutes. You only need to run this once per Colab session.

### Installing CRISPResso2

In [ ]:
!apt-get install -y git
!git clone https://github.com/pinellolab/CRISPResso2.git
%cd CRISPResso2
!pip install .

---
## Step 5: Prepare CRISPResso2 Batch File

This step generates a **batch settings file** that tells CRISPResso2 where each sample's FASTQ file is and what amplicon/guide to use.

**Update the parameters below:**

| Parameter | Description | Example |
|-----------|-------------|---------|
| `project_directory` | Path to your demultiplexed output folder from Step 3 | Auto-filled from Step 3 |
| `amplicon_seq` | Full reference amplicon sequence (the expected WT sequence) | `TCAGAGAG...CCTC` |
| `guide_seq` | 20-nt guide RNA sequence, **without PAM** | `TGCCTTCCATCGTCAGCACA` |

> **Important:** The `amplicon_seq` should be the complete expected sequence of your PCR product (after barcode trimming). The `guide_seq` should match exactly 20 nt of the amplicon where Cas9/Cas12 cuts.

In [ ]:
#@title Prepare CRISPResso2 Batch file

# ============================================================
# UPDATE THESE PARAMETERS for your experiment
# ============================================================
project_directory = demux_output  # From Step 3 above
amplicon_seq = 'TCAGAGAGCCTCAAGTCTCGGTCACCACAGCTCTACTGTCTATCTAGCTATCTATCAGCTGCCTTCCATCGTCAGCACACAAACTACACAAGAATCTGCTTATTTATAGGCCACCTTGTCCCTTCTACAATGGTGCAAGAACACACAAATTCACACACACACTGACACACACAAACCGATCGATTGATTGATTGATAATGAAGCAAGAGCAGGTCAGGATGGCAGTGCTCCTCATGCTCAACTGCTTCGTCAAGGCCACGGCGCCGCCGCCATGGCCGCCGTCGGCTTCGTCCGCCTCCTTCCTC'              #@param {type:"string"}
guide_seq = 'TGCCTTCCATCGTCAGCACA'                         #@param {type:"string"}
batch_settings_file = "os.path.join(WORK_DIR, 'batch_file.tsv')"  #@param {type:"string"}

# Build batch settings: CRISPResso2 needs 'name' and 'fastq_r1' per sample.
# amplicon_seq and guide_seq are passed as command-line args in Step 6.
batch_settings = pd.DataFrame(columns=['name', 'fastq_r1'])

ignore_dirs = ['.ipynb_checkpoints']

for sample_name in os.listdir(project_directory):
    if sample_name in ignore_dirs:
        continue

    sample_path = os.path.join(project_directory, sample_name)
    if os.path.isdir(sample_path):
        fastq_files = [f for f in os.listdir(sample_path) if f.endswith('.fastq')]
        if not fastq_files:
            print(f'No .fastq files found in {sample_path}. Skipping...')
            continue
        fastq_r1 = os.path.join(sample_path, fastq_files[0])

        new_row = pd.DataFrame({
            'name': [sample_name],
            'fastq_r1': [fastq_r1],
        })
        batch_settings = pd.concat([batch_settings, new_row], ignore_index=True)

batch_settings.to_csv(batch_settings_file, sep='\t', index=False)
print(f'Batch file created with {len(batch_settings)} samples: {batch_settings_file}')
batch_settings.head()

---
## Step 6: Run CRISPResso2 Batch Analysis

Execute CRISPResso2 in batch mode across all demultiplexed samples. This is the core analysis step.

**Full documentation:** [CRISPResso2 GitHub](https://github.com/pinellolab/CRISPResso2) | Run `!CRISPResso -h` in a cell to see all available parameters.

### Editing mode

Uncomment **only one** of the three command blocks below depending on your experiment type:

- **Genome editing (Cas nuclease):** Quantifies indels (insertions/deletions) at the cut site. This is the default.
- **Base editing (CBE/ABE):** Quantifies C-to-T or A-to-G conversions in the editing window. Requires `--base_editor_output`.
- **Prime editing:** Quantifies precise edits against an expected edited sequence. Requires `--prime_editing_pegRNA_extension_quantification_window_size`.

### Key parameters reference

| Parameter | Flag | Default | Description |
|-----------|------|---------|-------------|
| Amplicon sequence | `--amplicon_seq` | *required* | WT reference amplicon sequence |
| Guide RNA | `-g` | *required* | 20-nt guide sequence (without PAM) |
| Window center | `-wc` | -3 | Center of quantification window relative to the cut site (use -10 for Cas12) |
| Window size | `-w` | 1 | Size of the quantification window (bp); increase for broader analysis |
| Min. alignment score | `--min_average_read_quality` | 0 | Minimum average quality to keep a read |
| Ignore substitutions | `--ignore_substitutions` | False | If set, only count insertions/deletions as modifications (recommended for genome editing to get pure indel frequency) |
| Exclude bp from sides | `--exclude_bp_from_left` / `--exclude_bp_from_right` | 15 | Exclude N bp from each end to avoid primer artifacts |
| Base editor output | `--base_editor_output` | False | Enable base editing quantification mode |
| Prime edit pegRNA | `--prime_editing_pegRNA_extension_quantification_window_size` | 0 | Window for prime editing quantification |

> **Tip for genome editing:** Add `--ignore_substitutions` so that the reported modification percentage reflects **indel frequency only**, excluding any PCR/sequencing-introduced substitutions.

In [ ]:
#@title Run CRISPResso Batch
# ============================================================
# CRISPResso2 parameters -- UPDATE as needed
# Full docs: https://github.com/pinellolab/CRISPResso2
# Run !CRISPResso -h to see all available parameters
# ============================================================
batch_file = batch_settings_file  # From Step 5
window_center = -10  #@param {type:"integer"}
window_size = 20     #@param {type:"integer"}

# ============================================================
# GENOME EDITING (default) -- Cas nuclease indel quantification
# Uncomment the command below for standard genome editing analysis.
# --ignore_substitutions ensures modification % = indel frequency only.
# ============================================================
!CRISPRessoBatch \
    --batch_settings {batch_file} \
    --amplicon_seq {amplicon_seq} \
    -g {guide_seq} \
    -wc {window_center} \
    -w {window_size} \
    --ignore_substitutions

# ============================================================
# BASE EDITING (CBE or ABE) -- Uncomment below INSTEAD of above
# Quantifies C>T (CBE) or A>G (ABE) conversions in the window.
# Docs: https://github.com/pinellolab/CRISPResso2#base-editing
# ============================================================
# !CRISPRessoBatch \
#     --batch_settings {batch_file} \
#     --amplicon_seq {amplicon_seq} \
#     -g {guide_seq} \
#     -wc {window_center} \
#     -w {window_size} \
#     --base_editor_output

# ============================================================
# PRIME EDITING -- Uncomment below INSTEAD of above
# Requires the expected edited sequence for comparison.
# Docs: https://github.com/pinellolab/CRISPResso2#prime-editing
# ============================================================
# expected_edited_seq = 'YOUR_EXPECTED_EDITED_AMPLICON'  #@param {type:"string"}
# !CRISPRessoBatch \
#     --batch_settings {batch_file} \
#     --amplicon_seq {amplicon_seq} \
#     -g {guide_seq} \
#     -wc {window_center} \
#     -w {window_size} \
#     --expected_hdr_amplicon_seq {expected_edited_seq} \
#     --prime_editing_pegRNA_extension_quantification_window_size 5

---
## Step 7: Export Results

Zip and download the CRISPResso2 output directory for downstream analysis and visualization.

In [ ]:
import glob
from google.colab import files

# Recursively search for CRISPRessoBatch output directories
batch_output = glob.glob(
    os.path.join(WORK_DIR, '**', 'CRISPRessoBatch_on_*'),
    recursive=True
)

# Keep only directories, not zip files or other matching files
batch_output = [path for path in batch_output if os.path.isdir(path)]

if batch_output:
    print("Found CRISPRessoBatch output folders:")
    for i, path in enumerate(batch_output):
        print(f"{i}: {path}")

    # Use the first match
    output_dir = batch_output[0]
    zip_path = output_dir + '.zip'

    !zip -r "{zip_path}" "{output_dir}"

    print(f'Zipped: {zip_path}')
    files.download(zip_path)

else:
    print('No CRISPRessoBatch output found anywhere under WORK_DIR.')

---
## Step 8: Manuscript-style CRISPResso2 Figures

These three cells let  user point the notebook at generated CRISPResso2 output folders or a zipped CRISPResso2 result, then create:

1. violin plots comparing groups,
2. per-line editing efficiency plots,
3. mutation profile plots from CRISPResso effect-vector files.

The only required input is a `CRISPRessoBatch_on_*` folder or `.zip`. A metadata CSV is optional, but recommended for clean labels. If provided, it should contain columns such as `sample`, `target`, `group`, `construct`, and `line`.


In [ ]:
#@title 8A. Load CRISPResso2 results and make violin plots
# This cell accepts either a CRISPRessoBatch_on_* folder, a .zip file, or a comma-separated list of folders.
# Optional metadata CSV columns: sample, target, group, construct, line

import os, re, glob, zipfile, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn'])
    import seaborn as sns

try:
    from scipy import stats
except Exception:
    stats = None

warnings.filterwarnings('ignore')

try:
    display
except NameError:
    from IPython.display import display

if 'WORK_DIR' not in globals():
    WORK_DIR = '/content'

# -------------------------
# User inputs
# -------------------------
CRISPResso_input = ''  #@param {type:"string"}
upload_zip_now = False  #@param {type:"boolean"}
metadata_csv = ''  #@param {type:"string"}
output_dir = '/content/crispresso_manuscript_figures'  #@param {type:"string"}

sample_column = 'sample'  #@param {type:"string"}
target_column = 'target'  #@param {type:"string"}
group_column = 'group'  #@param {type:"string"}
construct_column = 'construct'  #@param {type:"string"}
line_column = 'line'  #@param {type:"string"}

group_order_text = 'with hormones,no hormones'  #@param {type:"string"}
construct_order_text = 'gR1,gR2,gR3,gR4,iSpyMac Cas9'  #@param {type:"string"}
group_colors_text = 'with hormones:#4C78A8,no hormones:#E15759'  #@param {type:"string"}

violin_grid_columns = 3  #@param {type:"integer"}
violin_panel_width = 3.7  #@param {type:"number"}
violin_panel_height = 3.1  #@param {type:"number"}
violin_style = 'violin + points'  #@param ["violin + points", "box + points", "points only"]
show_p_values_vs_control = True  #@param {type:"boolean"}
control_construct = 'iSpyMac Cas9'  #@param {type:"string"}
save_prefix = 'CRISPResso'  #@param {type:"string"}

Path(output_dir).mkdir(parents=True, exist_ok=True)

# -------------------------
# Small helpers
# -------------------------
def _split_csv_text(text):
    return [x.strip() for x in str(text).split(',') if x.strip()]

def _parse_color_map(text, fallback=None):
    fallback = fallback or {}
    out = dict(fallback)
    for item in _split_csv_text(text):
        if ':' in item:
            k, v = item.split(':', 1)
            out[k.strip()] = v.strip()
    return out

def _star(p):
    if p is None or np.isnan(p):
        return ''
    if p < 0.0001:
        return '****'
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def _uploaded_zip_if_requested():
    if not upload_zip_now:
        return []
    try:
        from google.colab import files
        uploaded = files.upload()
        return list(uploaded.keys())
    except Exception as e:
        print('Upload is available only in Colab. Error:', e)
        return []

def _unzip_if_needed(path):
    path = str(path).strip()
    if not path:
        return []
    if path.lower().endswith('.zip'):
        out = os.path.join(WORK_DIR, Path(path).stem)
        Path(out).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(path, 'r') as z:
            z.extractall(out)
        return [out]
    return [path]

def _find_batch_dirs(input_text):
    paths = []
    for uploaded in _uploaded_zip_if_requested():
        paths.extend(_unzip_if_needed(uploaded))
    for item in _split_csv_text(input_text):
        paths.extend(_unzip_if_needed(item))
    if not paths:
        paths = glob.glob(os.path.join(WORK_DIR, 'CRISPRessoBatch_on_*'))
    batch_dirs = []
    for path in paths:
        if os.path.isdir(path) and os.path.basename(path).startswith('CRISPRessoBatch_on_'):
            batch_dirs.append(path)
        elif os.path.isdir(path):
            batch_dirs.extend(glob.glob(os.path.join(path, '**', 'CRISPRessoBatch_on_*'), recursive=True))
    batch_dirs = sorted({os.path.abspath(x) for x in batch_dirs if os.path.isdir(x)})
    if not batch_dirs:
        raise FileNotFoundError('No CRISPRessoBatch_on_* folder was found. Set CRISPResso_input or upload a zip.')
    return batch_dirs

def _target_from_batch_dir(batch_dir):
    name = os.path.basename(batch_dir)
    name = re.sub(r'^CRISPRessoBatch_on_', '', name)
    name = re.sub(r'(_split)?_batch$', '', name)
    return name

def _infer_from_sample(sample, batch_dir):
    s = str(sample)
    target = _target_from_batch_dir(batch_dir)
    group = 'group 1'
    construct = 'sample'
    line = s

    upper = s.upper()
    if upper.startswith('NOH') or '_HF_' in upper or upper.startswith('HF'):
        group = 'no hormones'
    elif upper.startswith('H') or '_H_' in upper:
        group = 'with hormones'

    m = re.search(r'GR\s*([0-9]+)', upper)
    if m:
        construct = f'gR{m.group(1)}'
    elif 'CT' in upper or 'CONTROL' in upper:
        construct = 'iSpyMac Cas9'
    elif re.search(r'(^|[_-])5512([_-]|$)', upper):
        construct = 'iSpyMac Cas9'
    elif re.search(r'(^|[_-])5513([_-]|$)', upper):
        construct = 'gR2'

    m = re.search(r'(?:^|[_-])L?([0-9]+)$', s)
    if m:
        line = int(m.group(1))
    return target, group, construct, line

def _read_quantification(sample_dir):
    q = os.path.join(sample_dir, 'CRISPResso_quantification_of_editing_frequency.txt')
    if not os.path.exists(q):
        return None
    df = pd.read_csv(q, sep='\t')
    if df.empty:
        return None
    row = df.iloc[0].to_dict()
    def getnum(*names):
        for name in names:
            if name in row:
                return pd.to_numeric(row[name], errors='coerce')
        return np.nan
    return {
        'unmodified_percent': getnum('Unmodified%', 'Unmodified'),
        'indel_percent': getnum('Modified%', 'Modified'),
        'reads_input': getnum('Reads_in_input'),
        'reads_aligned': getnum('Reads_aligned'),
        'insertions': getnum('Insertions'),
        'deletions': getnum('Deletions'),
        'substitutions': getnum('Substitutions'),
    }

def _load_metadata(path):
    if not str(path).strip():
        return None
    meta = pd.read_csv(path)
    rename = {
        sample_column: 'sample',
        target_column: 'target',
        group_column: 'group',
        construct_column: 'construct',
        line_column: 'line',
    }
    rename = {k: v for k, v in rename.items() if k in meta.columns}
    meta = meta.rename(columns=rename)
    if 'sample' not in meta.columns:
        raise ValueError('Metadata CSV must contain the sample column specified above.')
    keep = [c for c in ['sample', 'target', 'group', 'construct', 'line'] if c in meta.columns]
    return meta[keep].drop_duplicates('sample')

def collect_crispresso_results(batch_dirs, metadata_path=''):
    rows = []
    for batch_dir in batch_dirs:
        for sample_dir in sorted(glob.glob(os.path.join(batch_dir, 'CRISPResso_on_*'))):
            if not os.path.isdir(sample_dir):
                continue
            sample = os.path.basename(sample_dir).replace('CRISPResso_on_', '', 1)
            quant = _read_quantification(sample_dir)
            if quant is None:
                continue
            target, group, construct, line = _infer_from_sample(sample, batch_dir)
            rows.append({
                'sample': sample,
                'sample_dir': sample_dir,
                'batch_dir': batch_dir,
                'target': target,
                'group': group,
                'construct': construct,
                'line': line,
                **quant,
            })
    results = pd.DataFrame(rows)
    if results.empty:
        raise ValueError('No sample-level CRISPResso quantification files were found.')

    meta = _load_metadata(metadata_path)
    if meta is not None:
        results = results.merge(meta, on='sample', how='left', suffixes=('', '_meta'))
        for col in ['target', 'group', 'construct', 'line']:
            mcol = f'{col}_meta'
            if mcol in results.columns:
                results[col] = results[mcol].combine_first(results[col])
                results = results.drop(columns=[mcol])
    results['indel_percent'] = pd.to_numeric(results['indel_percent'], errors='coerce')
    results = results.dropna(subset=['indel_percent'])
    return results

# -------------------------
# Load data
# -------------------------
batch_dirs = _find_batch_dirs(CRISPResso_input)
results_df = collect_crispresso_results(batch_dirs, metadata_csv)
results_df.to_csv(os.path.join(output_dir, f'{save_prefix}_CRISPResso_summary.csv'), index=False)

print(f'Loaded {len(results_df)} samples from {len(batch_dirs)} CRISPRessoBatch folder(s).')
print('Targets:', ', '.join(map(str, sorted(results_df['target'].unique()))))
print('Groups:', ', '.join(map(str, sorted(results_df['group'].unique()))))
print('Summary CSV:', os.path.join(output_dir, f'{save_prefix}_CRISPResso_summary.csv'))
display(results_df.head())

# -------------------------
# Violin plot
# -------------------------
sns.set_theme(style='white', font='DejaVu Sans')
group_colors = _parse_color_map(group_colors_text, {'with hormones': '#4C78A8', 'no hormones': '#E15759'})
group_order = [g for g in _split_csv_text(group_order_text) if g in set(results_df['group'])]
group_order += [g for g in sorted(results_df['group'].dropna().unique()) if g not in group_order]
construct_order = [c for c in _split_csv_text(construct_order_text) if c in set(results_df['construct'])]
construct_order += [c for c in sorted(results_df['construct'].dropna().unique()) if c not in construct_order]
targets = sorted(results_df['target'].dropna().unique())

target_cols = max(1, min(violin_grid_columns, len(targets)))
target_rows = int(math.ceil(len(targets) / target_cols))
n_rows = max(1, len(group_order) * target_rows)
n_cols = target_cols
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(violin_panel_width * n_cols, violin_panel_height * n_rows),
    sharey=True,
    squeeze=False,
)
for ax in axes.flat:
    ax.axis('off')

for ri, group in enumerate(group_order):
    for ti, target in enumerate(targets):
        row = ri * target_rows + (ti // target_cols)
        col = ti % target_cols
        ax = axes[row, col]
        ax.axis('on')
        sub = results_df[(results_df['target'] == target) & (results_df['group'] == group)].copy()
        present_constructs = [c for c in construct_order if c in set(sub['construct'])]
        if sub.empty or not present_constructs:
            ax.axis('off')
            continue
        color = group_colors.get(group, '#4C78A8')
        if violin_style == 'violin + points':
            sns.violinplot(data=sub, x='construct', y='indel_percent', order=present_constructs,
                           ax=ax, color=color, inner=None, cut=0, linewidth=0.9)
        elif violin_style == 'box + points':
            sns.boxplot(data=sub, x='construct', y='indel_percent', order=present_constructs,
                        ax=ax, color=color, width=0.55, fliersize=0, linewidth=0.9)
        sns.stripplot(data=sub, x='construct', y='indel_percent', order=present_constructs,
                      ax=ax, color='#333333', size=3.2, alpha=0.65, jitter=0.15)

        if show_p_values_vs_control and stats is not None:
            ctrl = control_construct if control_construct in present_constructs else present_constructs[-1]
            ctrl_vals = sub.loc[sub['construct'] == ctrl, 'indel_percent'].dropna().values
            for xi, con in enumerate(present_constructs):
                if con == ctrl:
                    continue
                vals = sub.loc[sub['construct'] == con, 'indel_percent'].dropna().values
                if len(vals) >= 2 and len(ctrl_vals) >= 2:
                    _, pval = stats.mannwhitneyu(vals, ctrl_vals, alternative='two-sided')
                    ymax = max(np.max(vals), np.max(ctrl_vals))
                    ax.text(xi, min(103, ymax + 7), _star(pval), ha='center', va='bottom', fontsize=8, fontweight='bold')

        ax.set_title(str(target), fontsize=11, fontstyle='italic')
        ax.set_xlabel('')
        ax.set_ylabel('Indel Frequency (%)' if col == 0 else '')
        ax.set_ylim(-5, 110)
        ax.tick_params(axis='x', labelrotation=0)
        for label in ax.get_xticklabels():
            label.set_fontsize(8)
        sns.despine(ax=ax)

# Add group color legend
handles = [plt.Rectangle((0, 0), 1, 1, color=group_colors.get(g, '#888888')) for g in group_order]
fig.legend(handles, group_order, loc='upper center', ncol=max(1, len(group_order)), frameon=False, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout(rect=[0, 0, 1, 0.96])
violin_path = os.path.join(output_dir, f'{save_prefix}_violin_comparison.png')
fig.savefig(violin_path, dpi=400, bbox_inches='tight')
plt.show()
print('Saved:', violin_path)

In [ ]:
#@title 8B. Per-line editing efficiency plot
# Run Cell 8A first. This cell uses results_df created above.

import os, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------
# User inputs
# -------------------------
panel_by = 'construct'  #@param ["construct", "group", "target"]
filter_group = 'ALL'  #@param {type:"string"}
filter_construct = 'ALL'  #@param {type:"string"}
filter_target = 'ALL'  #@param {type:"string"}
target_colors_text = 'OsGN1a:#90EE90,OsGS3:#FFA07A,OsROC5:#87CEEB'  #@param {type:"string"}
efficiency_grid_columns = 1  #@param {type:"integer"}
efficiency_panel_width = 10.0  #@param {type:"number"}
efficiency_panel_height = 2.8  #@param {type:"number"}
show_zygosity_thresholds = True  #@param {type:"boolean"}
bar_edge_color = '#4D4D4D'  #@param {type:"string"}
save_prefix = 'CRISPResso'  #@param {type:"string"}

if 'results_df' not in globals():
    raise RuntimeError('Run Cell 8A first so results_df is available.')
if 'output_dir' not in globals():
    output_dir = '/content/crispresso_manuscript_figures'
Path(output_dir).mkdir(parents=True, exist_ok=True)

def _split_csv_text(text):
    return [x.strip() for x in str(text).split(',') if x.strip()]

def _parse_color_map(text, fallback=None):
    fallback = fallback or {}
    out = dict(fallback)
    for item in _split_csv_text(text):
        if ':' in item:
            k, v = item.split(':', 1)
            out[k.strip()] = v.strip()
    return out

def _natural_key(x):
    parts = re.split(r'(\d+)', str(x))
    return [int(p) if p.isdigit() else p.lower() for p in parts]

def _apply_filter(df, col, value):
    if str(value).strip().upper() == 'ALL':
        return df
    return df[df[col].astype(str) == str(value).strip()]

plot_df = results_df.copy()
plot_df = _apply_filter(plot_df, 'group', filter_group)
plot_df = _apply_filter(plot_df, 'construct', filter_construct)
plot_df = _apply_filter(plot_df, 'target', filter_target)
if plot_df.empty:
    raise ValueError('No rows left after filters. Check filter_group, filter_construct, and filter_target.')

target_colors = _parse_color_map(target_colors_text)
def _target_color(target, idx):
    default = plt.cm.Set2(idx % 8)
    return target_colors.get(str(target), default)

panels = sorted(plot_df[panel_by].dropna().unique(), key=_natural_key)
targets = sorted(plot_df['target'].dropna().unique(), key=_natural_key)
n_cols = max(1, min(efficiency_grid_columns, len(panels)))
n_rows = int(math.ceil(len(panels) / n_cols))
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(efficiency_panel_width * n_cols, efficiency_panel_height * n_rows),
    sharey=True,
    squeeze=False,
)

for pi, panel in enumerate(panels):
    ax = axes[pi // n_cols, pi % n_cols]
    sub = plot_df[plot_df[panel_by] == panel].copy()
    lines = sorted(sub['line'].dropna().unique(), key=_natural_key)
    x = np.arange(len(lines), dtype=float)
    width = min(0.82 / max(1, len(targets)), 0.28)

    for ti, target in enumerate(targets):
        vals = []
        for line in lines:
            v = sub[(sub['line'] == line) & (sub['target'] == target)]['indel_percent']
            vals.append(float(v.iloc[0]) if len(v) else np.nan)
        offset = (ti - (len(targets) - 1) / 2) * width
        ax.bar(x + offset, vals, width=width * 0.92, color=_target_color(target, ti),
               edgecolor=bar_edge_color, linewidth=0.35, label=str(target), alpha=0.95)

    if show_zygosity_thresholds:
        for y in [10, 30, 70]:
            ax.axhline(y, color='#9E9E9E', linestyle='--', linewidth=0.55, zorder=0)
        right = len(lines) - 0.45 if lines else 0
        ax.text(right, 5, 'WT', va='center', ha='left', fontsize=8)
        ax.text(right, 20, 'Chimeric', va='center', ha='left', fontsize=8)
        ax.text(right, 50, 'Monoallelic', va='center', ha='left', fontsize=8)
        ax.text(right, 85, 'Biallelic', va='center', ha='left', fontsize=8)

    ax.set_title(str(panel), fontsize=11)
    ax.set_ylabel('Indel Frequency (%)')
    ax.set_xlabel('T0 lines')
    ax.set_ylim(0, 105)
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in lines], fontsize=8)
    for sp in ax.spines.values():
        sp.set_linewidth(0.6)
        sp.set_color('#555555')

for j in range(len(panels), n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis('off')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=max(1, len(labels)), frameon=True, fontsize=8, bbox_to_anchor=(0.5, 1.02))
fig.text(0.5, 0.01, 'Zygosity categories: WT < 10% | Chimeric: 10%-30% | Monoallelic: 30%-70% | Biallelic: >70%',
         ha='center', va='bottom', fontsize=8,
         bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='#444444', linewidth=0.6))
plt.tight_layout(rect=[0, 0.05, 1, 0.96])

eff_path = os.path.join(output_dir, f'{save_prefix}_per_line_efficiency.png')
fig.savefig(eff_path, dpi=400, bbox_inches='tight')
plt.show()
print('Saved:', eff_path)

In [ ]:
#@title 8C. Mutation profile plots from CRISPResso effect vectors
# Run Cell 8A first. This cell uses Effect_vector_deletion.txt and Effect_vector_insertion.txt.

import os, re, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

# -------------------------
# User inputs
# -------------------------
profile_mode = 'average by target/construct/group'  #@param ["average by target/construct/group", "one plot per individual line"]
profile_target = 'ALL'  #@param {type:"string"}
profile_construct = 'ALL'  #@param {type:"string"}
profile_group = 'ALL'  #@param {type:"string"}
profile_group_by = 'target,construct,group'  #@param {type:"string"}
profile_grid_columns = 2  #@param {type:"integer"}
profile_panel_width = 9.5  #@param {type:"number"}
profile_panel_height = 2.7  #@param {type:"number"}
profile_window_half_width = 18  #@param {type:"integer"}
manual_start_position = 0  #@param {type:"integer"}
manual_end_position = 0  #@param {type:"integer"}
deletion_color = '#4C78A8'  #@param {type:"string"}
insertion_color = '#E15759'  #@param {type:"string"}
scatter_color = '#333333'  #@param {type:"string"}
show_scatter_points = True  #@param {type:"boolean"}
save_prefix = 'CRISPResso'  #@param {type:"string"}

if 'results_df' not in globals():
    raise RuntimeError('Run Cell 8A first so results_df is available.')
if 'output_dir' not in globals():
    output_dir = '/content/crispresso_manuscript_figures'
Path(output_dir).mkdir(parents=True, exist_ok=True)

def _split_csv_text(text):
    return [x.strip() for x in str(text).split(',') if x.strip()]

def _apply_filter(df, col, value):
    if str(value).strip().upper() == 'ALL':
        return df
    return df[df[col].astype(str) == str(value).strip()]

def _read_effect_vector(sample_dir, event):
    path = os.path.join(sample_dir, f'Effect_vector_{event}.txt')
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep='\t', comment='#', header=None, names=['pos', 'effect'])
    return df.dropna().assign(pos=lambda d: d['pos'].astype(int), effect=lambda d: d['effect'].astype(float))

def _read_amplicon_and_guide(sample_dir):
    log = os.path.join(sample_dir, 'CRISPResso_RUNNING_LOG.txt')
    if not os.path.exists(log):
        return '', ''
    txt = Path(log).read_text(errors='ignore')
    amp = re.search(r'--amplicon_seq\s+([ACGTNacgtn]+)', txt)
    guide = re.search(r'--guide_seq\s+([ACGTNacgtn,]+)', txt)
    if guide is None:
        guide = re.search(r'\s-g\s+([ACGTNacgtn,]+)', txt)
    return (amp.group(1).upper() if amp else ''), (guide.group(1).split(',')[0].upper() if guide else '')

def _sample_profile(row):
    sd = row['sample_dir']
    d = _read_effect_vector(sd, 'deletion')
    i = _read_effect_vector(sd, 'insertion')
    if d is None or i is None:
        return None
    max_pos = int(max(d['pos'].max(), i['pos'].max()))
    del_arr = np.zeros(max_pos, dtype=float)
    ins_arr = np.zeros(max_pos, dtype=float)
    del_arr[d['pos'].values - 1] = d['effect'].values
    ins_arr[i['pos'].values - 1] = i['effect'].values
    amp, guide = _read_amplicon_and_guide(sd)
    return {'del': del_arr, 'ins': ins_arr, 'amplicon': amp, 'guide': guide}

def _aggregate_profiles(sub):
    profiles = []
    amplicon, guide = '', ''
    max_len = 0
    for _, row in sub.iterrows():
        p = _sample_profile(row)
        if p is None:
            continue
        profiles.append(p)
        max_len = max(max_len, len(p['del']), len(p['ins']))
        amplicon = amplicon or p['amplicon']
        guide = guide or p['guide']
    if not profiles:
        return None
    dmat = np.zeros((len(profiles), max_len), dtype=float)
    imat = np.zeros((len(profiles), max_len), dtype=float)
    for r, p in enumerate(profiles):
        dmat[r, :len(p['del'])] = p['del']
        imat[r, :len(p['ins'])] = p['ins']
    return {
        'n': len(profiles),
        'x': np.arange(1, max_len + 1),
        'mean_del': dmat.mean(axis=0),
        'mean_ins': imat.mean(axis=0),
        'sem_del': dmat.std(axis=0, ddof=1) / np.sqrt(len(profiles)) if len(profiles) > 1 else np.zeros(max_len),
        'sem_ins': imat.std(axis=0, ddof=1) / np.sqrt(len(profiles)) if len(profiles) > 1 else np.zeros(max_len),
        'raw_del': dmat,
        'raw_ins': imat,
        'amplicon': amplicon,
        'guide': guide,
    }

def _window(profile):
    x = profile['x']
    if manual_start_position > 0 and manual_end_position > 0:
        return manual_start_position, manual_end_position
    guide = profile.get('guide', '')
    amp = profile.get('amplicon', '')
    center = None
    if guide and amp and guide in amp:
        guide_start = amp.find(guide) + 1
        guide_end = guide_start + len(guide) - 1
        center = guide_end - 3
    if center is None:
        total = profile['mean_del'] + profile['mean_ins']
        center = int(np.argmax(total) + 1) if len(total) else 1
    return max(1, center - profile_window_half_width), min(int(x.max()), center + profile_window_half_width)

def _plot_profile(ax, profile, title):
    xmin, xmax = _window(profile)
    keep = (profile['x'] >= xmin) & (profile['x'] <= xmax)
    x = profile['x'][keep]
    yd = profile['mean_del'][keep]
    yi = profile['mean_ins'][keep]
    yed = profile['sem_del'][keep]
    yei = profile['sem_ins'][keep]
    ymax = max(20, float(np.nanmax(yd + yi) * 1.2) if len(x) else 20)
    ymax = min(100, ymax)
    ax.set_ylim(0, ymax)
    ax.set_xlim(xmin - 0.5, xmax + 0.5)

    ax.bar(x - 0.22, yd, width=0.42, color=deletion_color, edgecolor='white', linewidth=0.15, label='Deletion', zorder=2)
    ax.bar(x + 0.22, yi, width=0.42, color=insertion_color, edgecolor='white', linewidth=0.15, label='Insertion', zorder=2)
    if profile['n'] > 1:
        ax.errorbar(x - 0.22, yd, yerr=np.minimum(yed, np.maximum(ymax - yd, 0)), fmt='none', ecolor='#666666', elinewidth=0.55, capsize=1.2, zorder=4)
        ax.errorbar(x + 0.22, yi, yerr=np.minimum(yei, np.maximum(ymax - yi, 0)), fmt='none', ecolor='#666666', elinewidth=0.55, capsize=1.2, zorder=4)
    if show_scatter_points and profile['n'] > 1:
        rng = np.random.default_rng(123)
        for matrix, offset in [(profile['raw_del'][:, keep], -0.22), (profile['raw_ins'][:, keep], 0.22)]:
            for row in matrix:
                jitter = rng.uniform(-0.12, 0.12, size=len(x))
                ax.scatter(x + offset + jitter, row, s=10, color=scatter_color, alpha=0.25, linewidths=0, zorder=3)

    amp = profile.get('amplicon', '')
    guide = profile.get('guide', '')
    if amp and len(amp) >= xmax:
        labels = list(amp[xmin - 1:xmax])
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=7.5, fontfamily='monospace')
        if guide and guide in amp:
            guide_start = amp.find(guide) + 1
            guide_end = guide_start + len(guide) - 1
            cut = guide_end - 3
            if xmin <= cut <= xmax:
                ax.axvline(cut + 0.5, color='#8f8f8f', linestyle='--', linewidth=0.8, zorder=1)
            pam_start, pam_end = guide_end + 1, min(guide_end + 4, len(amp))
            for pos, lbl in zip(x, ax.get_xticklabels()):
                if pam_start <= pos <= pam_end:
                    lbl.set_color('#CC0000')
                elif guide_start <= pos <= guide_end:
                    lbl.set_color('#1f4e99')
            if xmin <= pam_end and xmax >= pam_start:
                ax.add_patch(Rectangle((max(pam_start, xmin) - 0.5, -0.14), min(pam_end, xmax) - max(pam_start, xmin) + 1, 0.11,
                                       fill=False, edgecolor='#CC0000', linewidth=1.1,
                                       transform=ax.get_xaxis_transform(), clip_on=False, zorder=5))
    else:
        ax.set_xlabel('Amplicon position')

    ax.set_ylabel('Indel position (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.text(0.995, 0.985, f'n={profile["n"]}', transform=ax.transAxes, ha='right', va='top', fontsize=8)
    ax.legend(loc='upper left', fontsize=7.5, frameon=False, ncols=2)
    for sp in ax.spines.values():
        sp.set_linewidth(0.6)
        sp.set_color('#777777')

plot_df = results_df.copy()
plot_df = _apply_filter(plot_df, 'target', profile_target)
plot_df = _apply_filter(plot_df, 'construct', profile_construct)
plot_df = _apply_filter(plot_df, 'group', profile_group)
if plot_df.empty:
    raise ValueError('No rows left after profile filters.')

if profile_mode == 'one plot per individual line':
    group_cols = ['sample']
else:
    group_cols = [c for c in _split_csv_text(profile_group_by) if c in plot_df.columns]
    if not group_cols:
        group_cols = ['target']

panels = []
for key, sub in plot_df.groupby(group_cols, dropna=False):
    if not isinstance(key, tuple):
        key = (key,)
    title = ' | '.join(f'{col}: {val}' for col, val in zip(group_cols, key))
    prof = _aggregate_profiles(sub)
    if prof is not None:
        panels.append((title, prof))

if not panels:
    raise ValueError('No Effect_vector_deletion/insertion files were found for the selected samples.')

n_cols = max(1, min(profile_grid_columns, len(panels)))
n_rows = int(math.ceil(len(panels) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(profile_panel_width * n_cols, profile_panel_height * n_rows), squeeze=False)
for i, (title, prof) in enumerate(panels):
    _plot_profile(axes[i // n_cols, i % n_cols], prof, title)
for j in range(len(panels), n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis('off')
plt.tight_layout()
profile_path = os.path.join(output_dir, f'{save_prefix}_mutation_profiles.png')
fig.savefig(profile_path, dpi=400, bbox_inches='tight')
plt.show()
print('Saved:', profile_path)

---
## Step 9: Interpreting Results & Downstream Visualization

### What CRISPResso2 outputs

The batch output directory (`CRISPRessoBatch_on_*`) contains a folder for each sample with:

| File | Description |
|------|-------------|
| `CRISPResso_quantification_of_editing_frequency.txt` | Overall editing rates (% modified, unmodified) |
| `Alleles_frequency_table.zip` | Detailed allele frequency table (every unique sequence and its count) |
| `CRISPResso_mapping_statistics.txt` | Read mapping and quality metrics |
| `*.html` / `*.png` | Publication-ready plots (indel size distribution, allele plots, etc.) |

### Classifying T0 plant genotypes

For CRISPR-edited T0 plants, we classify editing outcomes based on indel frequency thresholds:

| Genotype | Indel Frequency | Interpretation |
|----------|----------------|----------------|
| Wild-type (WT) | < 10% | No editing detected |
| Chimeric | 10% - 30% | Somatic editing (mixed cell populations) |
| Monoallelic | 30% - 70% | One allele edited |
| Biallelic | > 70% | Both alleles edited |

### Visualization options

1. **CRISPResso2 built-in figures:** Open the HTML files in a browser for interactive plots
2. **Python (matplotlib/seaborn):** Import the `.txt` summary files into pandas for custom plotting

---

## Acknowledgments
- **CRISPResso2:** [Pinello Lab](https://github.com/pinellolab/CRISPResso2) ([Clement et al., 2019](https://doi.org/10.1038/s41587-019-0032-3))
- **Hi-TOM barcoding system:** [Liu et al., 2019](https://doi.org/10.1007/s11427-019-9570-x)
- **fastp:** [Chen et al., 2018](https://doi.org/10.1093/bioinformatics/bty560)

---

**Contact:** Innocent Byiringiro ([ibyiring@umd.edu](mailto:ibyiring@umd.edu)) | [Qi Lab](https://www.yipingqi.com/), University of Maryland, College Park